In [0]:
import pandas as pd

In [0]:
df_ratings_pd =pd.read_csv("/Volumes/movie_catalog/movie_schema/movie_vol/User_ratings.gzip", compression="gzip", parse_dates=["Review_date"])

In [0]:
df_ratings = spark.createDataFrame(df_ratings_pd)
display(df_ratings.show(5))

+------+-------+------+-------------------+
|UserID|MovieID|Rating|        Review_date|
+------+-------+------+-------------------+
|     1|   1193|     5|2000-12-31 20:12:40|
|     1|    661|     3|2000-12-31 20:35:09|
|     1|    914|     3|2000-12-31 20:32:48|
|     1|   3408|     4|2000-12-31 20:04:35|
|     1|   2355|     5|2001-01-06 21:38:11|
+------+-------+------+-------------------+
only showing top 5 rows


In [0]:
df_ratings.printSchema()

root
 |-- UserID: long (nullable = true)
 |-- MovieID: long (nullable = true)
 |-- Rating: long (nullable = true)
 |-- Review_date: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import col, lit,year,month,lpad,concat,StringType
df_ratings = df_ratings.withColumn("Rev_year",year(col("Review_date"))).withColumn("Rev_months",lpad(month(col("Review_date")),2,"0"))
df_ratings = df_ratings.withColumn("Review_ym",concat(col("Rev_year").cast(StringType()),col("Rev_months").cast(StringType())) )
display(df_ratings.sort("Review_ym").show(5))

+------+-------+------+-------------------+--------+----------+---------+
|UserID|MovieID|Rating|        Review_date|Rev_year|Rev_months|Review_ym|
+------+-------+------+-------------------+--------+----------+---------+
|  5952|    589|     4|2000-04-30 22:38:54|    2000|        04|   200004|
|  5952|   3005|     4|2000-04-30 22:12:30|    2000|        04|   200004|
|  5952|    590|     4|2000-04-30 22:42:22|    2000|        04|   200004|
|  5952|   3037|     5|2000-04-30 22:41:22|    2000|        04|   200004|
|  5952|    900|     4|2000-04-30 22:29:05|    2000|        04|   200004|
+------+-------+------+-------------------+--------+----------+---------+
only showing top 5 rows


In [0]:
df_ratings.select("Review_ym").distinct().count()

35

In [0]:
from pyspark.sql.functions import count,avg
df_ratings.groupby("Rev_year","Rev_months").agg(count("Rating").alias("Rating_count"),avg("Rating").alias("Avg_Rating")).orderBy("Rev_year","Rev_months").show()

+--------+----------+------------+------------------+
|Rev_year|Rev_months|Rating_count|        Avg_Rating|
+--------+----------+------------+------------------+
|    2000|        04|       11619| 3.565108873397022|
|    2000|        05|       67725| 3.611487633813215|
|    2000|        06|       54500| 3.636073394495413|
|    2000|        07|       91987|3.6246969680498333|
|    2000|        08|      180332|3.5741687554066943|
|    2000|        09|       53179| 3.616634385753775|
|    2000|        10|       41297| 3.610940262004504|
|    2000|        11|      291258| 3.571846953560074|
|    2000|        12|      112894|3.5841320176448703|
|    2001|        01|       18059| 3.542499584694612|
|    2001|        02|        8056| 3.554121151936445|
|    2001|        03|        6089|3.5312859254393167|
|    2001|        04|        5195|  3.47218479307026|
|    2001|        05|        4933| 3.461382525846341|
|    2001|        06|        4979|3.4691705161679054|
|    2001|        07|       

In [0]:
%python
catalog = "movie_catalog"
schema = dbName = db = "movie_schema"
volume_name = "monthwise_rating_data"

spark.sql(f'CREATE CATALOG IF NOT EXISTS `{catalog}`')
spark.sql(f'USE CATALOG `{catalog}`')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`')
spark.sql(f'USE SCHEMA `{schema}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume_name}`')
volume_folder =  f"/Volumes/{catalog}/{db}/{volume_name}"

In [0]:
df_ratings.select("Rev_year").distinct().collect()

[Row(Rev_year=2000),
 Row(Rev_year=2003),
 Row(Rev_year=2001),
 Row(Rev_year=2002)]

In [0]:
df_to_write =df_ratings.select("MovieID","Rating","Review_ym").filter("Rev_year < 2003")
df_to_write_latter =df_ratings.select("MovieID","Rating","Review_ym").filter("Rev_year >= 2003")
print(df_to_write.count())
df_to_write_latter.count()

996866


3343

In [0]:
df_ratings.groupby("Rev_year").agg(count("Rating").alias("Count of Ratings")).show()

+--------+----------------+
|Rev_year|Count of Ratings|
+--------+----------------+
|    2000|          904791|
|    2003|            3343|
|    2001|           68037|
|    2002|           24038|
+--------+----------------+



In [0]:
#for i in df_to_write.collect():
#   print(row['Review_ym'])



#df_to_write.repartition(30).write.format("csv").mode("overwrite").option("header", "True").save(volume_folder)

In [0]:
df_ratings.show()

+------+-------+------+-------------------+--------+----------+---------+
|UserID|MovieID|Rating|        Review_date|Rev_year|Rev_months|Review_ym|
+------+-------+------+-------------------+--------+----------+---------+
|     1|   1193|     5|2000-12-31 20:12:40|    2000|        12|   200012|
|     1|    661|     3|2000-12-31 20:35:09|    2000|        12|   200012|
|     1|    914|     3|2000-12-31 20:32:48|    2000|        12|   200012|
|     1|   3408|     4|2000-12-31 20:04:35|    2000|        12|   200012|
|     1|   2355|     5|2001-01-06 21:38:11|    2001|        01|   200101|
|     1|   1197|     3|2000-12-31 20:37:48|    2000|        12|   200012|
|     1|   1287|     5|2000-12-31 20:33:59|    2000|        12|   200012|
|     1|   2804|     5|2000-12-31 20:11:59|    2000|        12|   200012|
|     1|    594|     4|2000-12-31 20:37:48|    2000|        12|   200012|
|     1|    919|     4|2000-12-31 20:22:48|    2000|        12|   200012|
|     1|    595|     5|2001-01-06 21:3

In [0]:
lst =list(df_to_write.select("Review_ym").distinct())
print(lst)


[Column<'Review_ym'>]


In [0]:
df_to_write.select("Review_ym").distinct()

DataFrame[Review_ym: string]

In [0]:
from pyspark.sql.functions import col
import os
import shutil
def lpad(value, length, pad_char):
    return str(value).rjust(length, str(pad_char))
def replace(value,sstring,tstring):
    return value.replace(sstring,tstring)
cnt =1 
for row in df_to_write.select("Review_ym").distinct().orderBy("Review_ym").collect():
    print(row['Review_ym'],df_to_write.filter(col("Review_ym") == row['Review_ym']).count(),cnt)
    df_to_write.filter(col("Review_ym") == row['Review_ym']).coalesce(1).write.format("csv").mode("overwrite").option("header", "True").save(volume_folder)
    #if cnt <10:
        #cnt = cnt +1
        #continue;
    for filename in os.listdir(volume_folder):
        if filename.endswith(".csv"):
            newfilename =replace(filename,"-00000-","-"+lpad(str(cnt),5,0)+"-")
            shutil.move(os.path.join(volume_folder,filename),os.path.join('/Volumes/movie_catalog/movie_schema/movie_user_ratings',newfilename))
    if cnt > 11:
        break;
    cnt = cnt +1    
    #shutil.rmtree(volume_folder) 


200004 11619 1
200005 67725 2
200006 54500 3
200007 91987 4
200008 180332 5
200009 53179 6
200010 41297 7
200011 291258 8
200012 112894 9
200101 18059 10
200102 8056 11
200103 6089 12


from pyspark.sql.functions import replace
filename ="""/Volumes/movie_catalog/movie_schema/movie_user_ratings/part-00000-tid-6008058980271264566-2173be1d-195c-419b-b451-83830226ec84-327-1-c000.csv"""
cnt = 1

def lpad(value, length, pad_char):
    return str(value).rjust(length, str(pad_char))
def replace(value,sstring,tstring):
    return value.replace(sstring,tstring)
print(lpad(str(cnt),5,0))
print(replace(filename,"-00000-","-"+lpad(str(cnt),5,0)+"-"))